In [1]:
%pip install ranx

  Using cached numba-0.65.1-cp312-cp312-macosx_12_0_arm64.whl.metadata (2.9 kB)
  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
  Using cached ir_datasets-0.5.11-py3-none-any.whl.metadata (12 kB)
  Using cached lz4-4.4.5-cp312-cp312-macosx_11_0_arm64.whl.metadata (3.8 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached llvmlite-0.47.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.0 kB)
  Using cached inscriptis-2.7.1-py3-none-any.whl.metadata (27 kB)
  Using cached trec_car_tools-2.6-py3-none-any.whl.metadata (640 bytes)
  Using cached warc3_wet-0.2.5-py3-none-any.whl.metadata (2.2 kB)
  Using cached warc3_wet_clueweb09-0.2.5-py3-none-any.whl
  Using cached zlib_state-0.1.12-cp312-cp312-macosx_10_13_universal2.whl
  Using cached ijson-3.5.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (23 kB)
  Using cached unlzw3-0.2.3-py3-none-any.whl.metadata (2.3 kB)
  Using cached lxml-6.0.4-cp312-cp312-macosx_10_13_universal2.whl.metadata (3.1 kB)
  

In [2]:
from ranx import Qrels, Run, evaluate

def evaluate_precision_at_k_with_ranx(qrels_dict, run_dict, k=3):
    print(f"📊 ranx로 Precision@{k} 평가를 시작합니다...")
    
    # 1. 파이썬 딕셔너리를 ranx 전용 객체로 변환
    qrels = Qrels(qrels_dict)
    run = Run(run_dict)
    
    # 2. 평가 지표 설정
    metrics = [f"precision@{k}"]
    
    # 3. 평가 수행
    try:
        results = evaluate(qrels, run, metrics=metrics)
        
        # [수정된 부분] 
        # 지표가 1개일 때는 results가 단순 숫자(float)로 나오고, 
        # 지표가 여러 개일 때는 딕셔너리(dict)로 나옵니다. 이를 모두 대응합니다.
        if isinstance(results, dict):
            final_score = results[f'precision@{k}']
        else:
            final_score = results # results 자체가 그냥 점수입니다.
            
        print("✅ 평가 완료!")
        print(f"💡 전체 평균 Precision@{k} 점수: {final_score:.4f}")
        
        return final_score

    except Exception as e:
        print(f"❌ 평가 중 오류가 발생했습니다: {e}")
        return None

# ==========================================
# 🚀 사용 예시 
# ==========================================

ground_truth_qrels = {
    "q1": {"doc1": 1, "doc2": 1},
    "q2": {"doc3": 1}
}

retrieved_run = {
    "q1": {"doc1": 0.99, "doc3": 0.85, "doc4": 0.70},
    "q2": {"doc1": 0.95, "doc3": 0.88, "doc5": 0.60}
}

# 함수 실행
evaluation_result = evaluate_precision_at_k_with_ranx(
    qrels_dict=ground_truth_qrels, 
    run_dict=retrieved_run, 
    k=3
)

📊 ranx로 Precision@3 평가를 시작합니다...


/Users/chaeyeonghwan/Desktop/gitHub/RFP-RAG-Extractor/venv/lib/python3.12/site-packages/ranx/metrics/precision.py:24: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  scores[i] = _precision(qrels[i], run[i], k, rel_lvl)


✅ 평가 완료!
💡 전체 평균 Precision@3 점수: 0.3333


In [3]:
%pip install nltk

  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [nltk]1/2 [nltk]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import nltk
import ssl

# 로컬(Mac/Windows) 환경에서 흔히 발생하는 SSL 인증서 다운로드 에러 방지
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context
    
# 최초 실행 시 형태소 분석(토큰화)을 위한 데이터를 다운로드합니다.
# NLTK 필수 데이터 다운로드 (최초 1회만 실행되며, 이미 있으면 알아서 스킵합니다)
print("다운로드를 시작합니다...")
nltk.download('punkt', quiet=False)
nltk.download('punkt_tab', quiet=False)
print("다운로드 완료!")

다운로드를 시작합니다...


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/chaeyeonghwan/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/chaeyeonghwan/nltk_data...


다운로드 완료!


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [7]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


def evaluate_generator_bleu(generated_answers, ground_truths):
    """
    NLTK를 이용하여 Generator가 생성한 텍스트의 BLEU 점수를 평가하는 함수입니다.
    
    Args:
        generated_answers (list of str): LLM이 생성한 답변 리스트
        ground_truths (list of str): 실제 완벽한 정답 리스트 (Reference)
        
    Returns:
        dict: 개별 점수 리스트와 전체 평균 BLEU 점수가 담긴 딕셔너리
    """
    print("📊 Generator BLEU 평가를 시작합니다...")
    
    bleu_scores = []
    # 짧은 문장에서도 점수가 0점이 나오지 않도록 스무딩(Smoothing) 기법 적용
    smoothie = SmoothingFunction().method1
    
    for gen_ans, truth in zip(generated_answers, ground_truths):
        # 1. 텍스트를 단어(토큰) 단위로 쪼갭니다.
        # 예: "복수의결권은 10년이다" -> ['복수의결권은', '10년이다']
        reference = [nltk.word_tokenize(truth)]
        candidate = nltk.word_tokenize(gen_ans)
        
        # 2. BLEU 점수 계산 (N-gram 겹침 정도를 수학적으로 계산)
        # weights=(0.5, 0.5)는 1-gram(단어 1개)과 2-gram(단어 2개 연속)까지만 주로 보겠다는 의미입니다.
        score = sentence_bleu(reference, candidate, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
        bleu_scores.append(score)
        
    # 3. 전체 평균 계산
    avg_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0
    
    print("✅ 평가 완료!")
    print(f"💡 전체 평균 BLEU 점수: {avg_bleu:.4f} (1.0에 가까울수록 완벽한 일치)")
    
    return {
        "individual_scores": bleu_scores,
        "average_score": avg_bleu
    }

# ==========================================
# 🚀 사용 예시 (Mock Data)
# ==========================================
# 사람이 작성해둔 실제 완벽한 정답 (Ground Truths)
sample_ground_truths = [
    "복수의결권주식은 1주당 최대 10개 한도로 발행할 수 있습니다.",
    "스톡옵션 부여 신고는 벤처확인종합관리시스템에서 가능합니다."
]

# 우리의 RAG 시스템(LLM)이 생성해낸 답변
sample_generated_answers = [
    "복수의결권주식은 1주당 최대 10개 한도로 발행이 가능합니다.", # 정답과 거의 똑같이 말함 (BLEU 높음)
    "스톡옵션을 신고하려면 중소벤처24 홈페이지를 이용하면 됩니다."    # 의미는 맞지만 단어가 완전히 다름 (BLEU 낮음)
]

# 함수 실행
results = evaluate_generator_bleu(
    generated_answers=sample_generated_answers,
    ground_truths=sample_ground_truths
)

# 상세 결과 확인
for i, score in enumerate(results["individual_scores"]):
    print(f" - 질문 {i+1} BLEU 점수: {score:.4f}")


📊 Generator BLEU 평가를 시작합니다...
✅ 평가 완료!
💡 전체 평균 BLEU 점수: 0.3133 (1.0에 가까울수록 완벽한 일치)
 - 질문 1 BLEU 점수: 0.5777
 - 질문 2 BLEU 점수: 0.0488
